# Integrated Notebook: JSON → VIDEO (UI + Backend)

> **Purpose:** A single, presentation-ready Jupyter Notebook that mirrors the logic and structure of your `final_solution.py` backend and `UI_moviePy.py` frontend. It is designed to be presented to teammates: contains explanatory markdown cells, and runnable code cells (kept in a single-file `.py`-style layout inside cells for clarity).

---

## Instructions for use

1. Place this notebook next to `scene_composition_agent_output.json`, `final_solution.py` assets, and your `.env` containing `GOOGLE_FONTS_API_KEY`.
2. Require a running virtual Python enviroment with all package installed.
3. Run cells sequentially. The notebook performs the following steps:
   - Load JSON project
   - Download assets (images, audio, fonts)
   - Render scenes in parallel (child processes)
   - Concatenate the final video

>**Below:** annotated cells for the two main files.

# **FILE: final_solution.py**
Core logic and rendering workflow

---

# Cell 1 — Imports & Environment

In [ ]:
# final_solution.py - core rendering backend
# Purpose: build videos from JSON scene descriptions using MoviePy, with
# parallel scene rendering, asset download/caching and safe error handling.

# Standard library utilities
import os          # filesystem paths and operations
import sys         # sometimes needed for path or platform-specific handling
import json        # read/write JSON (input scene description and index)
import re          # regex used for parsing Google Drive URLs / filenames
import shutil      # move/copy files, remove temp dirs
import tempfile    # for temporary directories if needed
import argparse    # CLI arg parsing for main()
import threading   # locks and thread pools for downloads
from dataclasses import dataclass, field  # lightweight data containers
from typing import List, Optional, Tuple

# Concurrency
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed

# System / diagnostics
import psutil      # CPU/memory info and process manipulation
import time
import platform
import subprocess
from datetime import datetime

# MoviePy (v2) pieces used to build & compose clips
from moviepy import (
    ImageClip,
    AudioFileClip,
    ColorClip,
    TextClip,
    CompositeVideoClip,
    concatenate_videoclips,
    VideoFileClip,
)
# Small FX helpers imported explicitly
from moviepy.video.fx.FadeIn import FadeIn
from moviepy.video.fx.FadeOut import FadeOut

# Helpers for downloads & fonts
import gdown       # download files from Google Drive
import requests    # HTTP requests (fonts fallback)
from dotenv import load_dotenv  # read GOOGLE_FONTS_API_KEY from .env

# Cell 2 — Data models (dataclasses)

In [ ]:
# Purpose: lightweight typed containers to mirror JSON structure.
# These make it easy to pass structured data around and to reconstruct
# objects from the JSON input.

@dataclass
class Animation:
    type: str
    startTime_sec: Optional[float] = None
    duration_sec: Optional[float] = None
    start_zoom: Optional[float] = None
    end_zoom: Optional[float] = None
    direction: Optional[str] = None

@dataclass
class Position:
    x: int
    y: int
    anchor: str

@dataclass
class Layer:
    layer_id: str
    type: str                   # 'image' | 'text' | 'color'
    content: Optional[str] = None
    url: Optional[str] = None
    font: Optional[str] = None
    size: Optional[int] = None
    color: Optional[str] = None
    position: Optional[Position] = None
    animation: Optional[Animation] = None

@dataclass
class Scene:
    scene_id: int
    slide_id: int
    audioUrl: Optional[str]
    audioDuration_sec: float
    layers: List[Layer] = field(default_factory=list)

@dataclass
class VideoMetadata:
    title: str
    resolution: str
    fps: int

@dataclass
class VideoProject:
    metadata: VideoMetadata
    scenes: List[Scene]

# Cell 3 — Asset dirs & index helper functions

In [ ]:
# Purpose: keep downloaded assets in ./assets/{audio,images,fonts}
# and keep a simple index (download_index.json) mapping drive file_id -> path
ASSET_ROOT = os.path.join(os.getcwd(), "assets")
ASSET_DIRS = {
    "audio": os.path.join(ASSET_ROOT, "audio"),
    "images": os.path.join(ASSET_ROOT, "images"),
    "fonts": os.path.join(ASSET_ROOT, "fonts"),
}
DOWNLOAD_INDEX_PATH = os.path.join(ASSET_ROOT, "download_index.json") # Create a index file for debugging using the assets download ID
_INDEX_LOCK = threading.Lock()  # guard access to the index file across threads

# Create asset folders and a blank download index if missing.
def ensure_asset_dirs():
    for p in ASSET_DIRS.values():
        os.makedirs(p, exist_ok=True)
    if not os.path.exists(DOWNLOAD_INDEX_PATH):
        try:
            os.makedirs(os.path.dirname(DOWNLOAD_INDEX_PATH), exist_ok=True)
            with open(DOWNLOAD_INDEX_PATH, "w") as f:
                json.dump({}, f)
        except Exception:
            pass

# Return JSON index mapping file_id -> local path, or {} if not available.
def _load_download_index():
    try:
        if os.path.exists(DOWNLOAD_INDEX_PATH):
            with open(DOWNLOAD_INDEX_PATH, "r") as f:
                return json.load(f)
    except Exception:
        pass
    return {}

# Atomic write of index file (write to temp then replace).
def _save_download_index(index: dict):
    tmp = DOWNLOAD_INDEX_PATH + ".tmp"
    with open(tmp, "w") as f:
        json.dump(index, f)
    os.replace(tmp, DOWNLOAD_INDEX_PATH)

# Cell 4 — Google Drive helpers & download_asset

In [ ]:
# Purpose: extract Google Drive file id and download using gdown.
# download_asset does: check index -> gdown download -> move into assets -> update index.

def extract_file_id(drive_url: str) -> Optional[str]:
    if not drive_url:
        return None
    match = re.search(r"/d/([a-zA-Z0-9_-]+)", drive_url)
    return match.group(1) if match else None

# Downloads Google Drive file (using gdown) and places it into ./assets/{kind}/.
# Uses a thread-safe download_index so repeated calls won't re-download assets.
# Returns local path or None on failure.
def download_asset(url: str, kind: str) -> Optional[str]:
    ensure_asset_dirs()
    file_id = extract_file_id(url)
    if not file_id:
        print(f"⚠️ download_asset: invalid drive url: {url}")
        return None

    # quick check: index -> existing file
    with _INDEX_LOCK:
        idx = _load_download_index()
        mapped = idx.get(file_id)
        if mapped and os.path.exists(mapped):
            return mapped

    uc_url = f"https://drive.google.com/uc?export=download&id={file_id}" # Google Drive download link template
    try:
        downloaded_path = gdown.download(uc_url, output=None, quiet=False)
    except Exception as e:
        print(f"⚠️ gdown failed for {url}: {e}")
        downloaded_path = None

    if not downloaded_path or not os.path.exists(downloaded_path):
        print(f"⚠️ download_asset: gdown didn't return a valid file for {url}")
        return None

    # move to asset folder (ensure unique name if collision)
    filename = os.path.basename(downloaded_path)
    dest_dir = ASSET_DIRS.get(kind, ASSET_ROOT)
    dest_path = os.path.join(dest_dir, filename)
    base, ext = os.path.splitext(filename)
    counter = 1
    while os.path.exists(dest_path):
        dest_filename = f"{base}_{counter}{ext}"
        dest_path = os.path.join(dest_dir, dest_filename)
        counter += 1

    try:
        shutil.move(downloaded_path, dest_path)
    except Exception as e:
        try:
            shutil.copy2(downloaded_path, dest_path)
            os.remove(downloaded_path)
        except Exception as ee:
            print(f"⚠️ Could not move/copy downloaded file: {downloaded_path} -> {dest_path}: {ee}")
            return None

    # update index
    with _INDEX_LOCK:
        idx = _load_download_index()
        idx[file_id] = dest_path
        try:
            _save_download_index(idx)
        except Exception as e:
            print(f"⚠️ Failed to update download index: {e}")

    print(f"📥 Downloaded {kind}: {dest_path}")
    return dest_path

# Lookup previously downloaded asset path by file_id, or search folder for matching file.
def get_local_asset_path(url: str, kind: str) -> Optional[str]:
    file_id = extract_file_id(url)
    if not file_id:
        return None
    idx = _load_download_index()
    path = idx.get(file_id)
    if path and os.path.exists(path):
        return path
    # last-chance: search folder for filename containing file_id
    folder = ASSET_DIRS.get(kind, ASSET_ROOT)
    if os.path.isdir(folder):
        for fn in os.listdir(folder):
            if file_id in fn:
                candidate = os.path.join(folder, fn)
                if os.path.exists(candidate):
                    return candidate
    return None

# Cell 5 — Fonts helpers

In [ ]:
# Purpose: obtain a TTF font path for rendering TextClip.
# Tries Google Fonts API (if API key provided), falls back to DejaVuSans download.

def fetch_fallback_font() -> str:
    ensure_asset_dirs()
    fallback_path = os.path.join(ASSET_DIRS["fonts"], "DejaVuSans.ttf")
    if not os.path.exists(fallback_path):
        print("Downloading fallback font DejaVuSans...")
        r = requests.get("https://github.com/dejavu-fonts/dejavu-fonts/raw/master/ttf/DejaVuSans.ttf") # This one is only a placeholder for now, the github link doesn't exit
        if r.status_code == 200:
            with open(fallback_path, "wb") as f:
                f.write(r.content)
    return fallback_path

def fetch_google_font_via_api(font_name: str, api_key: str) -> str:
    ensure_asset_dirs()
    api_url = f"https://www.googleapis.com/webfonts/v1/webfonts?key={api_key}" # Google Fonts Endpoint template
    try:
        r = requests.get(api_url)
        if r.status_code != 200:
            return fetch_fallback_font()
        data = r.json()
        family_entry = next((f for f in data.get("items", []) if f["family"].lower() == font_name.lower()), None)
        if not family_entry:
            return fetch_fallback_font()
        font_url = family_entry["files"].get("regular")
        if not font_url:
            return fetch_fallback_font()
        font_path = os.path.join(ASSET_DIRS["fonts"], f"{font_name.replace(' ', '_')}.ttf")
        if not os.path.exists(font_path):
            resp = requests.get(font_url)
            if resp.status_code == 200:
                with open(font_path, "wb") as f:
                    f.write(resp.content)
                print(f"📥 Downloaded font: {font_path}")
        return font_path
    except Exception as e:
        print(f"⚠️ Error fetching font {font_name}: {e}")
        return fetch_fallback_font()

# Public wrapper: use Google Fonts API if requested, else fallback font.
def get_font_path(font_name: Optional[str], api_key: str) -> str:
    return fetch_google_font_via_api(font_name, api_key) if font_name else fetch_fallback_font()

# Cell 6 — Animation helpers

In [ ]:
# Purpose: apply requested animations to individual clips before composition.
# - apply_animation_to_clip: fade, slide, start time offsets
# - apply_kenburns_to_image: gradual zooming (Ken Burns)

def apply_animation_to_clip(clip, layer: Layer, safe_duration: float, canvas_size: Tuple[int, int]):
    if not layer.animation:
        return clip.with_duration(safe_duration)
    anim = layer.animation
    start = anim.startTime_sec or 0.0
    effect_duration = float(anim.duration_sec) if anim.duration_sec else 0.0
    # set clip start and ensure it doesn't exceed safe_duration
    clip = clip.with_start(start).with_duration(safe_duration - start)
    anim_type = (anim.type or "").lower()
    if anim_type == "fadein":
        clip = clip.with_effects([FadeIn(duration=effect_duration)])
    elif anim_type == "fadeout":
        clip = clip.with_effects([FadeOut(duration=effect_duration)])
    elif anim_type == "slideinfromleft":
        # custom position function that moves clip from left into "final" x
        canvas_w, canvas_h = canvas_size
        final_x = layer.position.x if layer.position else (canvas_w - clip.w) / 2
        final_y = layer.position.y if layer.position else (canvas_h - clip.h) / 2
        start_x = -clip.w
        def pos_fn(t):
            progress = min(max(t / effect_duration, 0.0), 1.0) if effect_duration > 0 else 1.0
            x = start_x + progress * (final_x - start_x)
            return (x, final_y)
        clip = clip.with_position(pos_fn)
    return clip

def apply_kenburns_to_image(clip, anim: Animation, safe_duration: float):
    # Ken Burns implemented as a time-varying resize function
    start_zoom = anim.start_zoom or 1.0
    end_zoom = anim.end_zoom or 1.1
    def scale_fn(t):
        progress = min(max(t / safe_duration, 0.0), 1.0)
        return start_zoom + (end_zoom - start_zoom) * progress
    return clip.resized(scale_fn)

# Cell 7 — JSON loader

In [ ]:
# Purpose: load the project JSON and convert nested dicts into dataclass instances
def load_project_from_json(json_path: str) -> VideoProject:
    with open(json_path, "r") as f:
        data = json.load(f)
    metadata = VideoMetadata(**data["videoMetadata"])
    scenes = []
    for s in data["scenes"]:
        layers = []
        for l in s["layers"]:
            position = Position(**l["position"]) if "position" in l else None
            animation = Animation(**l["animation"]) if "animation" in l else None
            layer_dict = {**l}
            layer_dict["position"] = position
            layer_dict["animation"] = animation
            layers.append(Layer(**layer_dict))
        scenes.append(Scene(**{**s, "layers": layers}))
    return VideoProject(metadata=metadata, scenes=scenes)

# Cell 8 — parallel_download_assets

In [ ]:
# Purpose: download all referenced assets (audio/images/fonts) concurrently
# Uses ThreadPoolExecutor because downloads are I/O-bound.
def parallel_download_assets(project: VideoProject, api_key: str):
    ensure_asset_dirs()
    tasks = []
    results = {}
    with ThreadPoolExecutor(max_workers=8) as executor:
        for scene in project.scenes:
            if scene.audioUrl:
                tasks.append(executor.submit(download_asset, scene.audioUrl, "audio"))
            for layer in scene.layers:
                if layer.type == "image" and layer.url:
                    tasks.append(executor.submit(download_asset, layer.url, "images"))
                elif layer.type == "text" and layer.font:
                    tasks.append(executor.submit(fetch_google_font_via_api, layer.font, api_key))
        for f in as_completed(tasks):
            try:
                path = f.result()
                if path:
                    results[path] = True
            except Exception as e:
                print(f"⚠️ Asset download failed: {e}")
    print(f"✅ {len(results)} assets ready in {ASSET_ROOT}")
    return results

# Cell 9 — render_scene (core per-scene renderer)

In [ ]:
# Purpose: create a scene video clip by stacking layers, applying animations,
# ensuring every layer has been resized to the target resolution to avoid
# broadcast/shape errors when composing. Attach audio if present, then export.
def render_scene(scene: Scene, width: int, height: int, api_key: str, temp_dir: str):
    """
    Safe renderer: resizes every layer to (width,height), protects against
    broadcasting errors, and writes scene_{scene_id}.mp4 to temp_dir.
    """
    import os
    from moviepy import CompositeVideoClip, ImageClip, AudioFileClip, TextClip, ColorClip

    # determine duration (prefer audio duration if available)
    scene_duration = scene.audioDuration_sec
    audio_clip = None

    if scene.audioUrl:
        audio_path = get_local_asset_path(scene.audioUrl, "audio") or download_asset(scene.audioUrl, "audio")
        if audio_path:
            try:
                audio_clip = AudioFileClip(audio_path)
                scene_duration = min(scene_duration, audio_clip.duration)
            except Exception as e:
                print(f"[Warning] Could not load audio for scene {scene.scene_id}: {e}")

    EPSILON = 0.02
    safe_duration = max(0, scene_duration - EPSILON)
    layer_clips = []

    # Build each layer (image, color, text) with explicit resizing and animations
    for layer in scene.layers:
        try:
            if layer.type == "image" and layer.url:
                img_path = get_local_asset_path(layer.url, "images") or download_asset(layer.url, "images")
                if img_path:
                    # resize image clip explicitly to target resolution
                    img_clip = ImageClip(img_path).resize((width, height)).with_duration(safe_duration)
                    if layer.animation and (layer.animation.type or "").lower() == "kenburns":
                        img_clip = apply_kenburns_to_image(img_clip, layer.animation, safe_duration)
                    img_clip = apply_animation_to_clip(img_clip, layer, safe_duration, (width, height))
                    layer_clips.append(img_clip)
            elif layer.type == "color" and layer.color:
                rgb = tuple(int(layer.color.lstrip("#")[i:i + 2], 16) for i in (0, 2, 4))
                color_clip = ColorClip(size=(width, height), color=rgb).with_duration(safe_duration)
                color_clip = apply_animation_to_clip(color_clip, layer, safe_duration, (width, height))
                layer_clips.append(color_clip)
            elif layer.type == "text" and (layer.content or "").strip():
                font_path = get_font_path(layer.font, api_key)
                try:
                    txt_clip = TextClip(
                        text=layer.content,
                        font=font_path,
                        font_size=layer.size or 40,
                        color=layer.color or "white",
                        size=(width, None),  # allow wrapping horizontally
                        method="caption",
                    )
                except Exception as e:
                    print(f"[Warning] TextClip creation failed for '{layer.content[:30]}': {e}")
                    continue

                if layer.position:
                    txt_clip = txt_clip.with_position((layer.position.x, layer.position.y))

                txt_clip = txt_clip.with_duration(safe_duration)
                txt_clip = apply_animation_to_clip(txt_clip, layer, safe_duration, (width, height))
                txt_clip = txt_clip.resize(newsize=(width, None))
                layer_clips.append(txt_clip)
        except Exception as e:
            print(f"[Warning] Skipped layer due to error in scene {scene.scene_id}: {e}")

    # If no layers, use a blank color clip so composition still works
    if not layer_clips:
        print(f"[Warning] Scene {scene.scene_id} has no valid visual layers. Using blank background.")
        layer_clips = [ColorClip(size=(width, height), color=(0, 0, 0)).with_duration(safe_duration)]

    # Safety pass: ensure every clip equals the target resolution to avoid broadcast errors
    fixed_layers = []
    for clip in layer_clips:
        try:
            cw, ch = clip.size
            if (cw, ch) != (width, height):
                clip = clip.resize(newsize=(width, height))
        except Exception as e:
            print(f"[Warning] Could not read clip size: {e}")
        fixed_layers.append(clip)

    # Compose and attach audio (subclip to safe_duration)
    scene_clip = CompositeVideoClip(fixed_layers, size=(width, height)).with_duration(safe_duration)
    if audio_clip:
        try:
            scene_clip = scene_clip.with_audio(audio_clip.subclipped(0, safe_duration))
        except Exception as e:
            print(f"[Warning] Could not attach audio: {e}")

    # Export the scene file
    scene_out = os.path.join(temp_dir, f"scene_{scene.scene_id}.mp4")
    try:
        scene_clip.write_videofile(
            scene_out,
            fps=30,
            codec="libx264",
            audio_codec="aac",
            threads="auto"
        )
    except Exception as e:
        print(f"[Error] Failed to render scene {scene.scene_id}: {e}")
        raise
    finally:
        scene_clip.close()
        if audio_clip:
            audio_clip.close()

    return scene_out

# Cell 10 — build_video_from_project_parallel & main

In [ ]:
# Purpose: orchestrate downloading assets, rendering scenes in parallel (ProcessPool),
# concatenating scene videos, writing final output and writing a detailed log.
def build_video_from_project_parallel(project: VideoProject, api_key: str):
    width, height = map(int, project.metadata.resolution.split("x"))

    temp_dir = os.path.join(os.getcwd(), "temp")
    result_dir = os.path.join(os.getcwd(), "results")
    os.makedirs(temp_dir, exist_ok=True)
    os.makedirs(result_dir, exist_ok=True)

    start_time = datetime.now()
    timestamp_str = start_time.strftime("%Y-%m-%d_%H-%M-%S")

    log_data = {
        "start_time": start_time.isoformat(),
        "system": platform.platform(),
        "cpu_count": os.cpu_count(),
        "status": "started",
        "errors": [],
        "warnings": [],
        "scenes": {},
    }

    print("🧩 Downloading all assets to ./assets/ ...")
    assets = parallel_download_assets(project, api_key)
    log_data["asset_count"] = len(assets)

    start_perf = time.perf_counter()

    print(f"🎨 Rendering scenes in parallel (temp files in {temp_dir})...")
    scene_outputs = []
    scene_times = []

    # Render scenes concurrently (CPU work), collect outputs
    with ProcessPoolExecutor(max_workers=os.cpu_count() or 4) as executor:
        futures = {executor.submit(render_scene, s, width, height, api_key, temp_dir): s for s in project.scenes}
        for f in as_completed(futures):
            scene = futures[f]
            scene_id = scene.scene_id
            scene_start = time.perf_counter()
            try:
                out_path = f.result()
                elapsed_scene = time.perf_counter() - scene_start
                scene_times.append(elapsed_scene)
                scene_outputs.append(out_path)
                log_data["scenes"][f"scene_{scene_id}"] = {
                    "slide_id": scene.slide_id,
                    "duration_sec": scene.audioDuration_sec,
                    "elapsed_sec": round(elapsed_scene, 2),
                    "status": "completed"
                }
                print(f"✅ Finished scene {scene_id}: {os.path.basename(out_path)} ({elapsed_scene:.2f}s)")
            except Exception as e:
                elapsed_scene = time.perf_counter() - scene_start
                log_data["errors"].append({"scene_id": scene_id, "error": str(e)})
                print(f"❌ Scene {scene_id} failed: {e}")
                # continue or abort depending on your desired behavior (here: abort)
                raise

    # Concatenate in scene order and write final output
    def scene_key(p):
        m = re.search(r"scene_(\d+)", os.path.basename(p))
        return int(m.group(1)) if m else p
    scene_outputs_sorted = sorted(scene_outputs, key=scene_key)
    clips = [VideoFileClip(p) for p in scene_outputs_sorted]
    final_clip = concatenate_videoclips(clips, method="compose")
    final_out_path = os.path.join(result_dir, f"final_{timestamp_str}.mp4")
    final_clip.write_videofile(final_out_path, codec="libx264", fps=project.metadata.fps, audio_codec="aac", threads="auto")
    print(f"🎬 Final video saved to: {final_out_path}")

    # fill log_data with stats, save it, cleanup temp
    end_perf = time.perf_counter()
    elapsed_total = end_perf - start_perf
    avg_scene_time = sum(scene_times) / len(scene_times) if scene_times else 0
    cpu_percent = psutil.cpu_percent(interval=1)
    mem = psutil.virtual_memory()
    log_data.update({
        "end_time": datetime.now().isoformat(),
        "elapsed_seconds": round(elapsed_total, 2),
        "avg_scene_seconds": round(avg_scene_time, 2),
        "video_metadata": {
            "title": project.metadata.title,
            "resolution": project.metadata.resolution,
            "fps": project.metadata.fps,
            "scene_count": len(project.scenes)
        },
        "hardware": {
            "cpu_percent": cpu_percent,
            "memory_used_mb": round(mem.used / 1_048_576, 2),
            "memory_total_mb": round(mem.total / 1_048_576, 2),
            "gpu_info": "nvidia-smi lookup attempted"
        },
        "status": "completed",
        "final_video_path": final_out_path
    })

    log_path = os.path.join(result_dir, f"render_log_{timestamp_str}.json")
    with open(log_path, "w") as f:
        json.dump(log_data, f, indent=4)

    print(f"📝 Render log saved to: {log_path}")
    return final_out_path

# CLI entrypoint
def main():
    parser = argparse.ArgumentParser(description="Render video from JSON (parallel by default).")
    parser.add_argument("json_path", nargs="?", default="scene_composition_agent_output.json")
    parser.add_argument("--no-parallel", action="store_true", help="Disable parallel mode.")
    args = parser.parse_args()

    load_dotenv()
    api_key = os.getenv("GOOGLE_FONTS_API_KEY")
    if not api_key:
        raise ValueError("Missing GOOGLE_FONTS_API_KEY in .env")

    project = load_project_from_json(args.json_path)
    print("⚙️ Running in parallel mode..." if not args.no_parallel else "⚙️ Running single-threaded.")
    output = build_video_from_project_parallel(project, api_key)
    print(f"🎬 Done! Saved to {output}")

if __name__ == "__main__":
    main()

---

# **FILE: UI_moviePy.py** 
GUI orchestration for rendering workflow

# Cell 1 — Header & imports

In [ ]:
# UI_moviePy.py - Tkinter GUI wrapper for previewing JSON and running parallel renders.
# Purpose: provide a GUI that can import the JSON project, download assets, preview scenes,
# and launch a ProcessPool-based render while streaming per-scene logs into the UI.

import os, sys, threading, queue, time, math, json, shutil, traceback
from datetime import datetime
from pathlib import Path

# GUI toolkit
import tkinter as tk
from tkinter import ttk, filedialog, messagebox

# Image thumbnails via Pillow (PIL)
from PIL import Image, ImageTk

# Optional audio preview using pygame (if present)
try:
    import pygame
    PYGAME_AVAILABLE = True
except Exception:
    PYGAME_AVAILABLE = False

# Process / system control
import psutil

# Concurrency primitives for running child processes
from concurrent.futures import ProcessPoolExecutor, as_completed, CancelledError

# MoviePy used in the main process to concatenate final clips
from moviepy import VideoFileClip, concatenate_videoclips, ColorClip

# dotenv for reading GOOGLE_FONTS_API_KEY if needed
from dotenv import load_dotenv
load_dotenv()

# Try to import backend functions from final_solution (if available)
try:
    from final_solution import (
        load_project_from_json,
        parallel_download_assets,
        render_scene,            # used by child processes (pickled call)
        get_local_asset_path,
        ASSET_DIRS,
    )
    BACKEND_AVAILABLE = True
except Exception as e:
    print("⚠️ Could not import final_solution backend:", e)
    BACKEND_AVAILABLE = False

# Cell 2 — global state, logging helpers

In [ ]:
# Purpose: queues to pass log lines and UI events safely from worker threads/processes
LOG_QUEUE = queue.Queue()
UI_QUEUE = queue.Queue()

# Keep app-level state in a dict for easy cross-method access
app_state = {
    "project": None,
    "json_path": None,
    "assets_ready": False,
    "render_thread": None,
    "render_cancel": False,
    "executor": None,
    "temp_dir": None,
    "scene_outputs": [],
    "render_progress": {
        "completed": 0,
        "total": 0,
        "scene_times": [],
        "start_time": None
    }
}

def now_ts():
    """Return current time string for log prefixes."""
    return datetime.now().strftime("%H:%M:%S")

def log(msg: str):
    """Push a timestamped message into the LOG_QUEUE for the UI to display later."""
    LOG_QUEUE.put(f"[{now_ts()}] {msg}")

def ui_put(evt: str, payload):
    """Queue an event (evt name and payload) for the main UI loop to handle."""
    UI_QUEUE.put((evt, payload))

# Cell 3 — pygame audio helpers

In [ ]:
# Purpose: small wrappers to check/init pygame mixer and play/stop audio in UI preview.
def ensure_pygame():
    if not PYGAME_AVAILABLE:
        return False
    try:
        if not pygame.mixer.get_init():
            pygame.mixer.init()
        return True
    except Exception as e:
        log(f"⚠️ pygame init failed: {e}")
        return False

def play_audio_file(path: str):
    if not ensure_pygame():
        log("Audio playback unavailable (pygame).")
        return
    try:
        pygame.mixer.music.stop()
        pygame.mixer.music.load(path)
        pygame.mixer.music.play()
        log(f"▶ Playing audio {os.path.basename(path)}")
    except Exception as e:
        log(f"⚠️ Error playing audio: {e}")

def stop_audio_playback():
    if PYGAME_AVAILABLE and pygame.mixer.get_init():
        try:
            pygame.mixer.music.stop()
        except Exception:
            pass

# Cell 4 — child wrapper (picklable)

In [ ]:
# Purpose: Executed in a child process (via ProcessPoolExecutor).
# - Redirects stdout/stderr of child to per-scene log file
# - Calls final_solution.render_scene(...) — if that throws a broadcasting ValueError,
#   it will create a safe black fallback clip to avoid aborting the whole pipeline.
def _child_render_wrapper(scene, width, height, api_key, temp_dir, scene_log_path):
    import sys, traceback
    os.makedirs(os.path.dirname(scene_log_path), exist_ok=True)

    def write_log_line(s):
        try:
            with open(scene_log_path, "a", encoding="utf-8", errors="replace") as lf:
                lf.write(f"{s}\n")
        except Exception:
            pass

    start_msg = f"--- Child render start: scene {scene.scene_id} | target {width}x{height} ---"
    write_log_line(start_msg)

    # redirect stdout/stderr to log file so MoviePy/ffmpeg prints are captured
    try:
        with open(scene_log_path, "a", encoding="utf-8", errors="replace") as lf:
            old_out, old_err = sys.stdout, sys.stderr
            sys.stdout, sys.stderr = lf, lf
            try:
                # call the backend renderer (can use MoviePy/ffmpeg); outputs path to file
                out_path = render_scene(scene, width, height, api_key, temp_dir)
            finally:
                try:
                    sys.stdout.flush(); sys.stderr.flush()
                except Exception:
                    pass
                sys.stdout, sys.stderr = old_out, old_err

        write_log_line(f"--- Child render completed: scene {scene.scene_id} -> {out_path} ---")
        return {"scene_id": scene.scene_id, "out_path": out_path, "log_path": scene_log_path}

    except Exception as exc:
        # save traceback to child log, and detect shape/broadcasting ValueError to create fallback clip
        tb = traceback.format_exc()
        write_log_line("\n--- EXCEPTION IN CHILD ---")
        write_log_line(str(exc))
        write_log_line(tb)

        msg = str(exc).lower()
        if isinstance(exc, ValueError) and ("broadcast" in msg or "could not be broadcast" in msg or "shapes" in msg):
            write_log_line("--- Detected broadcasting ValueError — creating fallback blank scene to continue demo ---")
            try:
                dur = getattr(scene, "audioDuration_sec", None) or 3.0
                fallback_path = os.path.join(temp_dir, f"scene_{scene.scene_id}_fallback.mp4")
                color_clip = ColorClip(size=(width, height), color=(0, 0, 0)).with_duration(dur)
                color_clip.write_videofile(fallback_path, codec="libx264", fps=24, audio=False, threads=1)
                write_log_line(f"--- Fallback scene created at {fallback_path} ---")
                return {"scene_id": scene.scene_id, "out_path": fallback_path, "log_path": scene_log_path}
            except Exception as fallback_exc:
                write_log_line(f"--- Fallback creation failed: {fallback_exc} ---")
                # re-raise original exception after logging
                raise
        # non-broadcast error or fallback failed -> re-raise
        raise

# Cell 5 — UI class setup & preview tab

Due to the huge number of code lines and GPU instructions, these cells will be minimized up to method name with it's purpose.

In [ ]:
class UIRendererApp(tk.Tk):
    """
    Main Tkinter application for rendering JSON→Video projects.

    Tabs:
      • Preview tab — Import JSON, browse scenes, preview assets.
      • Render tab — Configure and launch parallel render jobs.
    """

    def __init__(self):
        """Initialize window, tabs, and periodic UI updates."""
        ...

    # --------------------------------------------------
    # PREVIEW TAB CONSTRUCTION & HANDLERS
    # --------------------------------------------------
    def _build_preview_tab(self, parent):
        """
        Construct the Preview tab UI:
          - "Import JSON" button
          - project info display (title/res/fps)
          - scrollable list of scene cards with thumbnail + text info
          - per-scene play-audio button if audio exists
        """
        ...

    def _populate_preview_list(self):
        """Render scene cards inside the scrollable frame based on current project."""
        ...

    def _create_scene_card(self, parent, scene):
        """
        Create a visual card for one scene:
          - thumbnail from image layer (first found)
          - labels for Scene ID, duration, audio
          - 'Play Audio' button (uses play_audio_file)
        """
        ...

    def _update_project_info(self):
        """Refresh title, resolution, fps fields from project metadata."""
        ...

    def _apply_resolution_fps(self):
        """
        Read resolution/fps entries from UI,
        update project.metadata before rendering.
        """
        ...

# --------------------------------A------------------
    # PROJECT FILE MANAGEMENT
    # --------------------------------------------------
    def _on_import_json(self):
        """
        Choose JSON via file dialog, load using load_project_from_json(),
        populate preview tab and enable buttons.
        """
        ...

    def _on_download_assets(self):
        """
        Launch parallel_download_assets() in background thread
        to prefetch images/audio/fonts; show status in log box.
        """
        ...

    def _on_remove_project(self):
        """Clear loaded project and reset Preview tab widgets."""
        ...


# Cell 6 — Render tab setup and controls
**the indentation ERRORS mean, these method is still inside the UIRendererApp class**

In [ ]:
    # --------------------------------------------------
    # RENDER TAB CONSTRUCTION & CONTROLS
    # --------------------------------------------------
    def _build_render_tab(self, parent):
        """
        Construct the Render tab UI:
          - 'Start Rendering' button
          - 'Stop' (graceful cancel)
          - 'SCRAM' (emergency kill)
          - progress bar, status labels, hardware info
          - log text box showing live scene logs
        """
        ...

    def _append_log(self, text):
        """Thread-safe method to append a line to the Text log widget."""
        ...

    def _set_status(self, text):
        """Update small status label under progress bar."""
        ...

    def _set_progress(self, completed, total):
        """Update progress bar value and percentage label."""
        ...

    def _toggle_render_buttons(self, running: bool):
        """
        Enable/disable Start/Stop/SCRAM buttons depending on current state.
        Prevents accidental double starts.
        """
        ...

# Cell 7 — Rendering orchestration methods
**the indentation ERRORS mean, these method is still inside the UIRendererApp class**

In [ ]:
    # --------------------------------------------------
    # RENDER ORCHESTRATION
    # --------------------------------------------------
    def _on_start_render(self):
        """
        User clicked 'Start Rendering':
          - Validate project loaded
          - Apply resolution/fps settings
          - Create temp/output folders
          - Launch background thread (_render_worker_parallel)
        """
        ...

    def _render_worker_parallel(self, project, width, height, api_key):
        """
        Worker thread function:
          - Creates ProcessPoolExecutor with N workers
          - Submits _child_render_wrapper per scene
          - Streams each scene's log file back into UI
          - Tracks progress & ETA
          - Concatenates all scenes into final video
          - Emits ui_put("render_finished") when done
        """
        ...

    def _on_stop_render(self):
        """
        Graceful cancel:
          - Set app_state["render_cancel"]=True
          - Shutdown executor with cancel_futures=True
          - UI remains responsive
        """
        ...

    # Importance so get detail explaination
    def _on_scram_pressed(self):
        """
        HARD EMERGENCY STOP — SCRAM
        Purpose:
          Immediately abort all rendering processes and reset the UI.
          Unlike Stop (which finishes the current scene gracefully),
          SCRAM kills everything now — even mid-encode.
        """
        # Ask for user confirmation before destructive action
        if not messagebox.askyesno("SCRAM Confirmation", "⚠️ Emergency stop all processes?"):
            return

        log("⚠️ SCRAM initiated — killing all rendering processes...")

        # Stop any audio playback
        stop_audio_playback()

        # Kill any child processes spawned by ProcessPoolExecutor
        try:
            parent_proc = psutil.Process(os.getpid())
            for child in parent_proc.children(recursive=True):
                try:
                    child.kill()
                except Exception:
                    pass
            log("🧨 All subprocesses terminated.")
        except Exception as e:
            log(f"⚠️ Error during SCRAM kill: {e}")

        # Shut down executor if active
        if app_state.get("executor"):
            try:
                app_state["executor"].shutdown(cancel_futures=True)
                log("🧹 Executor forcibly shut down.")
            except Exception as e:
                log(f"⚠️ Could not fully shut down executor: {e}")

        # Remove temporary render directory
        temp_dir = app_state.get("temp_dir")
        if temp_dir and os.path.isdir(temp_dir):
            try:
                shutil.rmtree(temp_dir)
                log(f"🧽 Temp directory cleared: {temp_dir}")
            except Exception as e:
                log(f"⚠️ Failed to remove temp dir: {e}")

        # Reset all render-related state
        app_state.update({
            "render_cancel": False,
            "executor": None,
            "render_thread": None,
            "temp_dir": None,
            "scene_outputs": [],
        })
        self._toggle_render_buttons(False)
        self._set_status("SCRAM complete. System reset.")
        log("✅ SCRAM complete — system reset.")

    def _on_render_finished(self, payload):
        """
        Called after render completes successfully.
          - Shows success message
          - Enables 'Open Folder' button
        """
        ...

# Cell 8 — Logging, metrics & UI polling
**the indentation ERRORS mean, these method is still inside the UIRendererApp class**

In [ ]:
    # --------------------------------------------------
    # LOGGING, METRICS, AND PERIODIC TASKS
    # --------------------------------------------------
    def _stream_file_to_log(self, file_path):
        """
        Continuously read a log file and feed lines into LOG_QUEUE.
        Runs in background thread per scene.
        """
        ...

    def _periodic_tasks(self):
        """
        Every ~200 ms (via Tk .after()):
          - Drain LOG_QUEUE → append to log box
          - Drain UI_QUEUE → handle render events
          - Refresh CPU/MEM info from psutil
          - Update progress bar
        """
        self.after(200, self._periodic_tasks)
        ...

    def _update_hardware_labels(self):
        """Update CPU %, RAM usage labels using psutil metrics."""
        ...

# Cell 9 — Application entry point

In [ ]:
# --------------------------------------------------
# APPLICATION ENTRY POINT
# --------------------------------------------------
def main():
    """Create and run the Tkinter app."""
    app = UIRendererApp()
    app.mainloop()

if __name__ == "__main__":
    main()

# 🧭 Development Notes & Outstanding Issues

### ⚙️ Rendering & Performance

| Issue | Description | Suggested Solution |
|-------|--------------|--------------------|
| **1. Final concatenation is single-threaded** | Each scene is rendered in parallel (via `ProcessPoolExecutor`), but `concatenate_videoclips()` still runs in the main process — causing a bottleneck, especially for many short clips. | • Replace MoviePy concatenation with **ffmpeg concat demuxer** (direct stream copy).<br>• Use MoviePy’s `concatenate_videoclips(method="chain")` if no composition overlap is needed.<br>• Optionally parallelize concatenation in batches then merge the batches. |
| **2. Scene render startup overhead** | For many short scenes, process startup costs dominate rendering time. | • Use a persistent pool with **warm worker processes** to reuse imports.<br>• Reduce worker count for small projects (`max_workers=min(len(scenes), os.cpu_count())`). |
| **3. Logging overhead** | Each process writes verbose logs to disk (`scene_X.log`), which slows IO. | • Add verbosity levels (INFO/WARN/ERROR).<br>• Stream to memory queue for UI in real time, writing only on completion. |
| **4. GUI responsiveness** | Tkinter remains responsive but freezes occasionally during large file writes. | • Move file concatenation into a thread (similar to scene rendering).<br>• Consider switching to **async Tk loops** or PySide2 if GUI complexity grows. |

---

### 🖋️ Text Rendering & Typography

| Issue | Description | Suggested Solution |
|-------|--------------|--------------------|
| **1. Text clipping / cutoff** | Text layers (`TextClip`) sometimes lose ascenders (tops of “T”, “h”) or descenders (bottoms of “g”, “y”). | • Use `method="caption"` with vertical padding or `size=(width, None)` + `align='center'`. <br>• Apply `.resize(newsize=(width, None))` *after* setting duration to preserve font box. |
| **2. Off-centered or truncated captions** | Center alignment varies by font metrics and image scaling. | • Use anchor-based positioning (`position.anchor='center'`), or dynamically compute `(canvas_w - text.w)/2` and `(canvas_h - text.h)/2`. <br>• Measure text bounding box after creation for precise placement. |
| **3. Font fallback inconsistency** | Missing fonts cause silent fallback to DejaVuSans with different spacing. | • Cache verified Google Fonts per project.<br>• Log font substitutions clearly. |
| **4. Multi-line wrapping limits** | Long text lines overflow at smaller resolutions. | • Enable dynamic font resizing: detect text width and reduce `font_size` accordingly.<br>• Add auto word-wrapping for long captions. |

---

### 🎞️ Animation & Visual Layer Effects

| Issue | Description | Suggested Solution |
|-------|--------------|--------------------|
| **1. Limited animation types** | Currently supports basic `fadein`, `fadeout`, `slideinfromleft`, `kenburns`. | • Add: `slideinfromright`, `slideup`, `slidedown`, `zoom`, `rotate`, `pulse`, `pan`. <br>• Define animation presets in JSON (`"animation": {"type": "slideup", "easing": "easeInOut"}`). |
| **2. Easing and timing** | All motion is linear and abrupt. | • Use easing functions (e.g., `tween` or custom lambda easing curves).<br>• Add `startDelay`, `repeat`, `bounce` options in `Animation` dataclass. |
| **3. Layer ordering ambiguity** | Layers are rendered in listed order; missing `z-index` control may cause unexpected overlaps. | • Add `z_index` to `Layer` class.<br>• Sort layers before composition: `sorted(scene.layers, key=lambda l: l.z_index)`. |
| **4. Transition between scenes** | Scene-to-scene transitions are abrupt cuts. | • Add crossfade between final seconds of scene N and start of scene N+1.<br>• Implement via MoviePy `concatenate_videoclips(..., method="compose", padding=-fade_duration)`. |

---

### 🔊 Audio Handling

| Issue | Description | Suggested Solution |
|-------|--------------|--------------------|
| **1. Mismatched audio duration** | Some audio clips are slightly longer than their scene video duration, causing truncation warnings. | • Always call `.subclipped(0, safe_duration)` or `.audio.set_duration(scene_duration)` before attach. |
| **2. No background music or crossfade** | Each scene uses only its voice/audio layer. | • Add an optional global background track with fade in/out per scene. |
| **3. Missing normalization** | Audio levels vary drastically between scenes. | • Normalize RMS before concatenation using `AudioClip.volumex()` or ffmpeg filters. |

---

### 🧩 Asset Management & Robustness

| Issue | Description | Suggested Solution |
|-------|--------------|--------------------|
| **1. Asset index concurrency** | Multiple threads write to `download_index.json` simultaneously. | • Use file lock or `threading.Lock()` around `_save_download_index()` (partial fix exists).<br>• Consider `sqlite` or `tinydb` for safe concurrent access. |
| **2. Temp directory persistence** | Old temp files not cleaned after crash. | • Auto-clean `temp/` and `results/` folders on startup. |
| **3. Missing asset verification** | Broken Google Drive links crash silently. | • Validate URLs before download; mark missing ones in log but continue rendering with placeholder color clip. |

---

### 🧰 Code Quality & Maintainability

| Issue | Description | Suggested Solution |
|-------|--------------|--------------------|
| **1. Hard-coded constants** | FPS, codec, fade duration, etc. are scattered across functions. | • Move to `config.py` or top-level constants. |
| **2. Mixed MoviePy v1/v2 APIs** | Some functions use old `clip.set_duration` or `.fx()` signatures. | • Standardize to MoviePy v2 methods (`with_duration`, `with_effects`). |
| **3. Error handling duplication** | Many similar try/except blocks in rendering logic. | • Refactor into decorators (`@safe_render`) or helper for consistent logging. |
| **4. UI-logic coupling** | Tkinter class manages both UI and backend orchestration. | • Split into `render_controller.py` (backend thread manager) and `ui_renderer.py` (pure GUI). |
| **5. Lack of tests** | No automated verification for JSON parsing or rendering correctness. | • Add lightweight pytest suite for JSON→Video pipeline sanity checks. |

---

### 🧠 Other Observations

| Area | Suggestion |
|-------|-------------|
| **Font downloading** | Add caching + checksum validation for downloaded fonts. |
| **Hardware utilization** | Investigate using GPU-accelerated ffmpeg if available (`h264_nvenc`). |
| **Progress reporting** | Stream render times per scene + total ETA to UI in real time. |
| **Cross-platform** | Test on macOS & Linux; file paths and ffmpeg binary may differ. |
| **User experience** | Add “Render Summary” dialog after completion (show duration, errors, output path). |

---

### ✅ Summary: Priority Roadmap

1. **High impact / low effort**
   - Fix text clipping & centering.
   - Switch to ffmpeg concat for faster merging.
   - Add safe asset fallback and cleanup.

2. **Medium effort**
   - Expand animation variety & transitions.
   - Improve SCRAM/Stop recovery flow.

3. **Long-term**
   - Decouple UI logic.
   - Implement async progress API.
   - Add GPU encoding & robust testing.
